### Dybdegående forståelse af C# Yield
Denne notebook tager dig gennem yield keywordet i C#. Vi starter med det grundlæggende, dykker ned i hukommelsesstyring og ser på den "magi", compileren udfører for at skabe tilstandsmaskiner (state machines).

Forudsætninger: Grundlæggende kendskab til C# og klasser.

1. Introduktion: Eager vs. Lazy Evaluation
Før vi forstår yield, skal vi forstå forskellen på at returnere en samling (Collection) og en iterator.

Det klassiske (Eager) approach
Når vi normalt returnerer data, beregner vi hele listen, gemmer den i hukommelsen, og returnerer den.

In [1]:
// CODE CELL 1
using System;
using System.Collections.Generic;
using System.Threading;

// Eager execution: Laver hele listen før returnering
List<int> GetNumbersEager(int count)
{
    var list = new List<int>();
    Console.WriteLine("Eager: Starter generering...");
    for (int i = 0; i < count; i++)
    {
        // Simulerer tungt arbejde
        // Thread.Sleep(100); 
        list.Add(i);
    }
    Console.WriteLine("Eager: Færdig med generering.");
    return list;
}

// Test
var numbers = GetNumbersEager(5);
foreach(var num in numbers) 
{
    Console.WriteLine($"Behandler: {num}");
}

Eager: Starter generering...
Eager: Færdig med generering.
Behandler: 0
Behandler: 1
Behandler: 2
Behandler: 3
Behandler: 4


### Yield (Lazy) approach
Med yield return udskyder vi udførelsen (Deferred Execution). Koden i metoden kører ikke, når metoden kaldes, men først når vi begynder at iterere over den (f.eks. i en foreach).

In [2]:
// CODE CELL 2
// Lazy execution: Returnerer én ad gangen
public IEnumerable<int> GetNumbersLazy(int count)
{
    Console.WriteLine("Lazy: Starter generering (første kald)...");
    for (int i = 0; i < count; i++)
    {
        Console.WriteLine($"Lazy: Genererer tal {i}");
        yield return i; // Her pauses metoden og værdien leveres
        Console.WriteLine($"Lazy: Genoptager efter tal {i}");
    }
    Console.WriteLine("Lazy: Færdig.");
}

// Læg mærke til output rækkefølgen her!
Console.WriteLine("Kalder metoden (ingen generering sker endnu)...");
var lazyNumbers = GetNumbersLazy(3);

Console.WriteLine("Starter foreach løkken nu...");
foreach(var num in lazyNumbers)
{
    Console.WriteLine($"   -> Forbruger modtog: {num}");
}

Kalder metoden (ingen generering sker endnu)...
Starter foreach løkken nu...
Lazy: Starter generering (første kald)...
Lazy: Genererer tal 0
   -> Forbruger modtog: 0
Lazy: Genoptager efter tal 0
Lazy: Genererer tal 1
   -> Forbruger modtog: 1
Lazy: Genoptager efter tal 1
Lazy: Genererer tal 2
   -> Forbruger modtog: 2
Lazy: Genoptager efter tal 2
Lazy: Færdig.


### Opgave 1: Forstå flowet
Kør koden i celle 2. Analyser outputtet nøje.

Spørgsmål/Opgave: Skriv en metode FilterOddNumbers der bruger yield return.

Den skal tage en IEnumerable<int> som input.

Den skal printe "Tjekker [tal]" for hvert tal.

Den skal kun yield return hvis tallet er ulige.

Lav en kæde: GetNumbersLazy(5) -> FilterOddNumbers(...) -> foreach.

Mål: Observer hvordan kontrollen hopper frem og tilbage mellem datakilden, filteret og foreach løkken. Data "pulles" (trækkes) gennem systemet, det bliver ikke pushet.

In [7]:
public IEnumerable<int> FilterOddNumbers(IEnumerable<int> nums){
    foreach(int number in nums) {
        Console.WriteLine($"Lazy: evaluer om {number} er odd?");
        if (number % 2 != 0){
            yield return number;
        }
        Console.WriteLine($"Lazy: Genoptager efter filtrering");
    }
}

public IEnumerable<int> GetNumbersLazy(int count)
{
    Console.WriteLine("Lazy: Starter generering (første kald)...");
    for (int i = 0; i < count; i++)
    {
        Console.WriteLine($"Lazy: Genererer tal {i}");
        yield return i; // Her pauses metoden og værdien leveres
        Console.WriteLine($"Lazy: Genoptager efter tal {i}");
    }
    Console.WriteLine("Lazy: Færdig.");
}

// Læg mærke til output rækkefølgen her!
Console.WriteLine("Kalder metoden (ingen generering sker endnu)...");
var lazyNumbers = GetNumbersLazy(5);
var onlyOddNumbers = FilterOddNumbers(lazyNumbers);

Console.WriteLine("Starter foreach løkken nu...");
foreach(var num in onlyOddNumbers)
{
    Console.WriteLine($"   -> Forbruger modtog: {num}");
}

Kalder metoden (ingen generering sker endnu)...
Starter foreach løkken nu...
Lazy: Starter generering (første kald)...
Lazy: Genererer tal 0
Lazy: evaluer om 0 er odd?
Lazy: Genoptager efter filtrering
Lazy: Genoptager efter tal 0
Lazy: Genererer tal 1
Lazy: evaluer om 1 er odd?
   -> Forbruger modtog: 1
Lazy: Genoptager efter filtrering
Lazy: Genoptager efter tal 1
Lazy: Genererer tal 2
Lazy: evaluer om 2 er odd?
Lazy: Genoptager efter filtrering
Lazy: Genoptager efter tal 2
Lazy: Genererer tal 3
Lazy: evaluer om 3 er odd?
   -> Forbruger modtog: 3
Lazy: Genoptager efter filtrering
Lazy: Genoptager efter tal 3
Lazy: Genererer tal 4
Lazy: evaluer om 4 er odd?
Lazy: Genoptager efter filtrering
Lazy: Genoptager efter tal 4
Lazy: Færdig.


### 2. Yield Break og Terminering
Ligesom vi har yield return til at levere en værdi, har vi yield break til at stoppe iterationen før tid. Dette fungerer som en return i en normal metode, men det afslutter sekvensen korrekt for modtageren.

In [8]:
// CODE CELL 4
public IEnumerable<string> GetNamesUntilStop()
{
    yield return "Anders";
    yield return "Mette";
    yield return "STOP"; // Vi returnerer denne, men vil stoppe bagefter? 
                         // Nej, lad os sige "STOP" er et signal om at afbryde.
    yield break;         // Her slutter sekvensen helt.
    yield return "Jens"; // Denne linje bliver aldrig nået (Unreachable code warning)
}

foreach(var name in GetNamesUntilStop())
{
    Console.WriteLine(name);
}

Anders
Mette
STOP


### 3. Under motorhjelmen: State Machines ⚙️
Dette er den vigtigste sektion. Når vi skriver yield, omskriver C# compileren din metode fuldstændigt. Den laver den ikke om til en metode, men til en Klasse.

Denne klasse implementerer IEnumerable<T> og IEnumerator<T>.

**Hvad sker der i memory?**

Tilstand (State): Compileren genererer et felt (typisk en int state), der holder styr på, hvor i koden vi nåede til sidst (før yield return).

Kontekst: Alle lokale variabler i din metode bliver til felter i denne genererede klasse, så de kan "huskes" mellem hvert kald til MoveNext().

Logikken bag IEnumerator
Når du kører en foreach, sker følgende i baggrunden

In [9]:
// Pseudo-kode for hvad foreach gør
var enumerator = dinCollection.GetEnumerator();
while (enumerator.MoveNext()) 
{
    var current = enumerator.Current;
    // Din kode her
}

Error: (2,18): error CS0103: The name 'dinCollection' does not exist in the current context

 ### Når MoveNext() kaldes på den genererede klasse:

1. Den kigger på state.

2. Den hopper (via en switch/goto logik) direkte ned til linjen efter den sidste yield return.

3. Den kører indtil næste yield return.

4. Den sætter Current til værdien.

5. Den returnerer true.

6. Hvis den når slutningen af metoden (eller yield break), returnerer den false.

🔥 Opgave 2: Den Manuelle Compiler (Svær!)
For at bevise at du forstår logikken, skal du nu "lege compiler". Du må IKKE bruge yield keywordet.

Opgave: Implementer en klasse RangeIterator manuelt, som virker præcis som denne metode ville gøre:

In [10]:
public IEnumerable<int> Range(int start, int count) {
    for (int i = 0; i < count; i++) {
        yield return start + i;
    }
}

Du skal implementere IEnumerable<int> og IEnumerator<int>. Du skal styre state (hvor langt er vi?) og current.

Tip: Du har typisk brug for to klasser (eller én der implementerer begge interfaces, men pas på med reset-logik). Det er lettest at lave en RangeEnumerable der returnerer en ny RangeEnumerator.

In [18]:
using System.Collections;

// Din manuelle implementering:
public class RangeEnumerable : IEnumerable<int>
{
    private int _start;
    private int _count;

    public RangeEnumerable(int start, int count)
    {
        _start = start;
        _count = count;
    }

    public IEnumerator<int> GetEnumerator()
    {
        // Returner din custom enumerator her
        return new RangeEnumerator(_start, _count);
    }

    IEnumerator IEnumerable.GetEnumerator() => GetEnumerator();
}



public class RangeEnumerator : IEnumerator<int>
{
    public int[] _range;
    int _position;

    public RangeEnumerator(int start, int count) 
    {
        _position = start;
        _range = new int[count];
    }

    public int Current => _range[_position];

    object IEnumerator.Current => Current;

    public bool MoveNext()
    {
        _position++;
        return (_position < _range.Length);
    }

    public void Reset() {
        _position=-1 ;
    }

    public void Dispose() {
    }
}

// Test din manuelle iterator:
var myRange = new RangeEnumerable(10, 5);
foreach(var i in myRange) {
    Console.WriteLine(i);
}
    

### 4. Hukommelse og Uendelige Sekvenser
En af de største fordele ved yield er hukommelseseffektivitet. Da vi ikke allokerer en liste, er hukommelsesforbruget $O(1)$ i stedet for $O(n)$, uanset hvor mange elementer sekvensen har.Dette gør det muligt at arbejde med uendelige sekvenser.

In [19]:
// CODE CELL 6
public IEnumerable<long> FibonacciSequence()
{
    long current = 0, next = 1;
    yield return current;
    yield return next;

    while (true) // Uendeligt loop! Farligt i normal kode, sikkert med yield (hvis brugt rigtigt)
    {
        long temp = current + next;
        current = next;
        next = temp;
        yield return temp;
    }
}

// Vi kan tage de første 10, selvom kilden er uendelig
foreach (var num in FibonacciSequence())
{
    if (num > 100) break; // VIGTIGT: Forbrugeren styrer stoppet
    Console.Write(num + " ");
}

0 1 1 2 3 5 8 13 21 34 55 89 

### 💣 Opgave 3: Memory Safety & Multiple Iteration (Ekspert niveau)
Der er to klassiske fælder ved IEnumerable:

**Multiple Enumeration:** Hvis du itererer over en yield metode to gange, kører koden to gange.

**Objekt Instansiering:** Selvom den ikke gemmer en liste, allokerer den en lille "statemachine class" på heappen hver gang GetEnumerator() kaldes.

Opgave: Vi har en log-fil simulator, der læser "uendeligt" (eller meget stort datasæt). Du skal skrive en metode AnalyzeLogs der modtager en IEnumerable<string>.

Kravene er:

1. Metoden skal finde den første log-linje der indeholder "ERROR".

2. Metoden skal derefter tælle hvor mange linjer der kommer efter error-linjen, indtil den møder "END".

3. Kritisk: Du må IKKE iterere forfra. Du må ikke kalde GetEnumerator() to gange (implicit via foreach). Du skal bruge den samme iterator til at løse begge problemer sekventielt, da streamen er forward-only (tænk netværksstream).

*Hint: Brug IEnumerator direkte i stedet for foreach.*

In [ ]:
// CODE CELL 7

// Simulator af datakilde (Rør ikke denne)
public IEnumerable<string> LogStream()
{
    yield return "INFO: Start";
    yield return "INFO: Working";
    yield return "ERROR: Something broke"; // Find denne
    yield return "DEBUG: Details 1";       // Tæl denne
    yield return "DEBUG: Details 2";       // Tæl denne
    yield return "END";                    // Stop tælling her
    yield return "INFO: Ignore this";
}

public void AnalyzeLogs(IEnumerable<string> logs)
{
    // Din kode her.
    // Brug logs.GetEnumerator() manuelt.
    // 1. Loop indtil "ERROR..." findes.
    // 2. Loop videre med SAMME enumerator og tæl indtil "END".
    // 3. Print antallet.
}

AnalyzeLogs(LogStream());

### 5. Konklusion & Best Practices

**Brug yield når:**

- Du returnerer store sekvenser og vil spare hukommelse.

- Du vil bygge "pipelines" (f.eks. LINQ Where, Select er bygget med yield).

- Du beregner værdier on-the-fly.

**Undgå yield når:**

- Du har brug for at tilgå data via index (data[5]).

- Dataen skal bruges flere gange (overvej ToList() for at cache resultatet).

- Du har brug for rekursion (C# optimerer ikke rekursiv yield godt, det kan give dybe stakke/langsom performance).